In [1]:
import re
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from sklearn.metrics.pairwise import cosine_similarity
import os
import random
from datasets import Dataset
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from scipy.spatial.distance import cosine

/home/mayur/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def parse_input_file(file_path):
    anchor, pros, cons = "", [], []
    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()
    for line in lines:
        line = line.strip()
        if line.startswith("Discussion Title:"):
            anchor = line.split("Discussion Title:", 1)[1].strip()
        elif "Pro:" in line:
            clean_line = re.sub(r"\[.*?\]\(.*?\)", "", line.split("Pro:", 1)[1].strip())
            pros.append(clean_line)
        elif "Con:" in line:
            clean_line = re.sub(r"\[.*?\]\(.*?\)", "", line.split("Con:", 1)[1].strip())
            cons.append(clean_line)
    return anchor, pros, cons

In [3]:
folder_path = "./Data" # This is folder path where all the training data is present. Please add your folder path here to work this code.
txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]
print(len(txt_files))
txt_files = random.sample(txt_files, min(1000, len(txt_files)))

988


In [ ]:
pairs = []

for file_name in txt_files:
    file_path = os.path.join(folder_path, file_name)
    anchor, pros, cons = parse_input_file(file_path)
    for pro in pros:
        pairs.append({
            "first": anchor,
            "second": pro,
            "label": 1
        })

    for pro in pros:
        for con in cons:
            pairs.append({
                "first": pro,
                "second": con,
                "label": 0
            })

In [ ]:
# As the data is very huge I want 200 random pairs with label 1 and label 0

label_1_items = [item for item in pairs if item['label'] == 1]
label_0_items = [item for item in pairs if item['label'] == 0]
selected_label_1 = random.sample(label_1_items, min(1000, len(label_1_items)))
selected_label_0 = random.sample(label_0_items, min(5000, len(label_0_items)))
input_data = selected_label_1 + selected_label_0
random.shuffle(input_data)
print("input data", len(input_data))
print("label_1_items", len(label_1_items))
print("label_0_items", len(label_0_items))
print("selected_label_1", len(selected_label_1))
print("selected_label_0", len(selected_label_0))

In [ ]:
from sklearn.model_selection import train_test_split
train_pairs, val_pairs = train_test_split(input_data, test_size=0.2, random_state=42)
print("No. of Training pairs : " + str(len(train_pairs)))
print("No. of Validation pairs : " + str(len(val_pairs)))

In [ ]:
train_sentences1 = [pair['first'] for pair in train_pairs]
train_sentences2 = [pair['second'] for pair in train_pairs]
train_labels = [pair['label'] for pair in train_pairs]

val_sentences1 = [pair['first'] for pair in val_pairs]
val_sentences2 = [pair['second'] for pair in val_pairs]
val_labels = [pair['label'] for pair in val_pairs]

In [ ]:
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
train_examples = [
    InputExample(texts=[pair['first'], pair['second']], label=float(pair['label']))
    for pair in train_pairs
]
batch_size = 16
train_dataloader = DataLoader(train_examples, shuffle = True, batch_size = batch_size)
print(len(train_examples))

In [ ]:
# train_loss = losses.ContrastiveLoss(model=model)
train_loss = losses.CosineSimilarityLoss(model = model)
# train_loss = losses.OnlineContrastiveLoss(model = model)

In [ ]:
os.environ["WANDB_DISABLED"] = "true"
os.environ["REPORT_TO"] = "none"

num_epochs = 10
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)

model.fit(
    train_objectives = [(train_dataloader, train_loss)],
    epochs = num_epochs,
    warmup_steps = warmup_steps,
    show_progress_bar = True
)

In [ ]:
fine_tuned_model = model
base_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

first_embeddings = fine_tuned_model.encode(val_sentences1, convert_to_numpy=True)
second_embeddings = fine_tuned_model.encode(val_sentences2, convert_to_numpy=True)

similarity_scores = [1 - cosine(first, second) for first, second in zip(first_embeddings, second_embeddings)]

roc_auc = roc_auc_score(val_labels, similarity_scores) # Used this for check how well the model is able to distinguish between similar and dissimilar pairs (0.5 = random guessing and 1.0 = excellent)
threshold = 0.5
predicted_labels = [1 if score > threshold else 0 for score in similarity_scores]

accuracy = accuracy_score(val_labels, predicted_labels)
f1 = f1_score(val_labels, predicted_labels)

print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score: {f1:.4f}")

'''
ROC-AUC Score: 0.9670
Accuracy: 0.9967
F1-Score: 0.8012
'''

In [ ]:
from datasets import load_dataset
valid_dataset = load_dataset("timchen0618/Kialo", split="validation")
test_dataset  = load_dataset("timchen0618/Kialo", split="test")
import pprint
test_data = [
    {
        'anchor': item['question'],
        'pros': item['perspectives'][0],
        'cons': item['perspectives'][1],
    }
    for item in test_dataset if item['type'] == 'binary'
]

# pprint.pprint(test_data)
claims = [item["anchor"] for item in test_data]
pros = [item["pros"] for item in test_data]
cons = [item["cons"] for item in test_data]

In [ ]:
# import torch.nn.functional as F

# cos_sim_claim_pro = []
# cos_sim_claim_con = []

# for sample in test_data:
#     emb_claim = model.encode(sample['anchor'], convert_to_tensor=True)
#     emb_pro = model.encode(sample['pros'], convert_to_tensor=True)
#     emb_con = model.encode(sample['cons'], convert_to_tensor=True)

#     cos_sim_pro = torch.nn.functional.cosine_similarity(emb_claim, emb_pro, dim=0).item()
#     cos_sim_con = torch.nn.functional.cosine_similarity(emb_claim, emb_con, dim=0).item()

#     cos_sim_claim_pro.append(cos_sim_pro)
#     cos_sim_claim_con.append(cos_sim_con)

# plt.figure(figsize=(10, 6))
# sns.kdeplot(cos_sim_claim_pro, label='Claim-Pro', shade=True)
# sns.kdeplot(cos_sim_claim_con, label='Claim-Con', shade=True)
# plt.xlabel('Cosine Similarity')
# plt.ylabel('Density')
# plt.title('KDE Plot: Cosine Similarity for Claim-Pro vs. Claim-Con')
# plt.legend()
# plt.show()

In [ ]:
claim_embeddings = base_model.encode(claims, convert_to_numpy=True)
pro_embeddings = base_model.encode(pros, convert_to_numpy=True)
con_embeddings = base_model.encode(cons, convert_to_numpy=True)

# The cosine function from SciPy computes the cosine distance, which is defined as: Cosine Distance = 1 − Cosine Similarity
base_similarities_pro = [1 - cosine(claim_emb, pro_emb) for claim_emb, pro_emb in zip(claim_embeddings, pro_embeddings)]
base_similarities_con = [1 - cosine(claim_emb, con_emb) for claim_emb, con_emb in zip(claim_embeddings, con_embeddings)]

In [ ]:
fined_tuned_claim_embeddings = model.encode(claims, convert_to_numpy = True)
fined_tuned_pro_embeddings = model.encode(pros, convert_to_numpy = True)
fined_tuned_con_embeddings = model.encode(cons, convert_to_numpy = True)

fine_tuned_similarities_pro = [1 - cosine(claim_emb, pro_emb) for claim_emb, pro_emb in zip(fined_tuned_claim_embeddings, fined_tuned_pro_embeddings)]
fine_tuned_similarities_con = [1 - cosine(claim_emb, con_emb) for claim_emb, con_emb in zip(fined_tuned_claim_embeddings, fined_tuned_con_embeddings)]

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.kdeplot(base_similarities_pro, label='Pro (Base)', fill=True, color="green")
sns.kdeplot(base_similarities_con, label='Con (Base)', fill=True, color="red")
plt.title('Base Model')
plt.xlabel('Cosine Similarity')
plt.legend()

# max_pro = max(fine_tuned_similarities_pro)
# max_con = max(fine_tuned_similarities_con)
# min_pro = min(fine_tuned_similarities_pro)
# min_con = min(fine_tuned_similarities_con)
# print("Max Claim-Pro similarity:", max_pro)
# print("Max Claim-Con similarity:", max_con)
# print("Min Claim-Pro similarity:", min_pro)
# print("Min Claim-Con similarity:", min_con)

plt.subplot(1, 2, 2)
sns.kdeplot(fine_tuned_similarities_pro, label='Pro (Fine-tuned)', fill=True, color="green")
sns.kdeplot(fine_tuned_similarities_con, label='Con (Fine-tuned)', fill=True, color="red")
plt.title('Fine-tuned Model')
plt.xlabel('Cosine Similarity')
plt.legend()

info_text = f"Epochs: {num_epochs}\nBatch Size: {16}\nTraining Pairs: {len(train_pairs)}"

plt.text(0.05, 0.95, info_text, transform=plt.gca().transAxes, fontsize=10,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))


plt.tight_layout()
plt.show()

In [ ]:
triplet_pairs = []

for file_name in txt_files:
    file_path = os.path.join(folder_path, file_name)
    anchor, pros, cons = parse_input_file(file_path)
    for pro in pros:
        for con in cons:
            triplet_pairs.append({
                "pro": pro,
                "con": con,
                "anchor": anchor
            })

In [ ]:
import pprint
pprint.pprint(triplet_pairs[0:3])

In [ ]:
random_triplets = random.sample(triplet_pairs, 5000)

train_examples = [
    InputExample(texts=[item['anchor'], item['pro'], item['con']])
    for item in random_triplets
]
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

In [ ]:
len(train_examples)

In [ ]:
train_loss = losses.TripletLoss(model=model)
os.environ["WANDB_DISABLED"] = "true"
os.environ["REPORT_TO"] = "none"
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
)
triplet_model = model

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

test_data = triplet_pairs[3000:3500]

# def compute_similarities(model, data):
#     sim_pro, sim_con = [], []
#     for item in data:
#         anchor_emb = model.encode(item['anchor'], convert_to_tensor=True).cpu().numpy()
#         pro_emb = model.encode(item['pro'], convert_to_tensor=True).cpu().numpy()
#         con_emb = model.encode(item['con'], convert_to_tensor=True).cpu().numpy()

#         sim_pro.append(cosine_similarity([anchor_emb], [pro_emb])[0][0])
#         sim_con.append(cosine_similarity([anchor_emb], [con_emb])[0][0])
#     return sim_pro, sim_con



def compute_similarities(model, data):
    sim_pro, sim_con = [], []
    for item in data:
        anchor_emb = model.encode(item['anchor'], convert_to_tensor=True)
        pro_emb = model.encode(item['pro'], convert_to_tensor=True)
        con_emb = model.encode(item['con'], convert_to_tensor=True)

        # Manual L2 normalization
        anchor_emb = anchor_emb / anchor_emb.norm(dim=1, keepdim=True)
        pro_emb = pro_emb / pro_emb.norm(dim=1, keepdim=True)
        con_emb = con_emb / con_emb.norm(dim=1, keepdim=True)

        sim_pro.append(cosine_similarity(anchor_emb, pro_emb).item())
        sim_con.append(cosine_similarity(anchor_emb, con_emb).item())
    return sim_pro, sim_con



# Base model similarities
# base_sim_pro, base_sim_con = compute_similarities(SentenceTransformer('all-mpnet-base-v2'), test_data)

# Fine-tuned model similarities
# ft_model = SentenceTransformer('./triplet-model')
ft_sim_pro, ft_sim_con = compute_similarities(triplet_model, test_data)

plt.figure(figsize=(12, 5))

# plt.subplot(1, 2, 1)
# sns.kdeplot(base_sim_pro, label='Pro (Base)', fill=True)
# sns.kdeplot(base_sim_con, label='Con (Base)', fill=True)
# plt.title('Base Model')
# plt.xlabel('Cosine Similarity')
# plt.legend()

plt.subplot(1, 2, 2)
sns.kdeplot(ft_sim_pro, label='Pro (Fine-tuned)', fill=True)
sns.kdeplot(ft_sim_con, label='Con (Fine-tuned)', fill=True)
plt.title('Fine-tuned Model')
plt.xlabel('Cosine Similarity')
plt.legend()

plt.tight_layout()
plt.show()